# 12 · Sampling-clock recovery

The transmitter's DAC and the receiver's ADC run on separate oscillators.
Their rates differ by a few to a hundred parts per million (ppm), so the
receiver takes slightly more or fewer than `sps` samples per transmitted
symbol, and the symbol instants drift through the record.  Frame
synchronization ([11](11-frame-synchronization-and-timing.ipynb)) finds one
timing offset for the whole capture; it cannot follow a drift.  This
notebook covers:

1. the **sampling-clock offset** and what it does to a record;
2. the **Gardner timing-error detector**, classical and on the signal power;
3. **tracking** the clock with a loop and **resampling** onto the symbol
   grid, or removing a known offset.

It builds on the matched filter of
[04](04-pulse-shaping-and-matched-filtering.ipynb) and the resampling of
[05](05-multirate-and-resampling.ipynb).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import commkit as ck
from commkit import RRC, Constellation
from commkit.backend import to_device
from commkit.timing import Gardner

DEVICE = "gpu" if ck.backend.is_cupy_available() else "cpu"
print(DEVICE)

ck.plotting.apply_default_theme()
C16 = Constellation.qam(16)


def host(a):
    return to_device(a, "cpu")


def link(rolloff, ppm, num_symbols=2**17, delay=0.37):
    """16-QAM at 32 GBd, 2 samples per symbol, Es/N0 = 20 dB, a fractional
    delay and an ADC clock off by ppm, through the matched filter."""
    tx = ck.generate(
        C16,
        num_symbols=num_symbols,
        symbol_rate=32e9,
        sps=2,
        pulse=RRC(rolloff=rolloff),
        rng=1,
        device=DEVICE,
    )
    rx = ck.impairments.apply_awgn(tx, esn0_db=20, rng=2)
    rx = ck.timing.fft_fractional_delay(rx, delay=delay)
    rx = ck.impairments.apply_sampling_clock_offset(rx, offset_ppm=ppm)
    return ck.filtering.matched_filter(rx)

## 1. The sampling-clock offset

`apply_sampling_clock_offset` resamples the waveform as an ADC whose clock
runs at `fs · (1 + ε)` while the samples keep the nominal rate: with
`ε = 50 ppm` the receiver takes `2 · (1 + 5·10⁻⁵)` samples per symbol, and
after 2¹⁷ symbols the symbol instants have moved by `2¹⁷ · ε ≈ 6.6` symbol
periods.  Sliced at the nominal symbol rate, the first symbols are close to
their eye's centre and the constellation is clean; a quarter of a symbol
period of drift later (5000 symbols) the samples sit between the symbols.

In [ ]:
drifting = link(rolloff=0.1, ppm=50.0, delay=0.0)
sliced = ck.multirate.decimate_to_symbol_rate(drifting)
fig, axes = plt.subplots(
    1, 2, figsize=np.multiply(plt.rcParams["figure.figsize"], (2, 1))
)
ck.plotting.plot_constellation(
    sliced.samples[:1000], ax=axes[0], title="Symbols 0-1000"
)
ck.plotting.plot_constellation(
    sliced.samples[5000:6000], ax=axes[1], title="Symbols 5000-6000"
)
plt.show()

## 2. The Gardner detector

At two samples per symbol, Gardner's detector compares two successive
symbol samples `y_{k-1}`, `y_k` with the sample halfway between them `m_k`:

$$e_k = \mathrm{Re}\{m_k\,(y_{k-1} - y_k)^*\}.$$

At a transition the midpoint sits on the zero crossing when the timing is
right; sampled late it has already moved towards `y_k`, and `e` turns
negative.  Averaged over the data, `e` against the timing error `τ` is a
sinusoid, the **S-curve**, whose slope at zero is the detector gain `K_d`.

It needs excess bandwidth: the transitions of a pulse with a small rolloff
carry little timing information, and the classical S-curve shrinks with the
rolloff.  The **power** form applies the same comparison to `|x|²`,

$$e_k = -|m_k|^2\,(|y_{k-1}|^2 - |y_k|^2),$$

whose clock tone survives small rolloffs.  Below, both S-curves measured by
hand on records delayed by known amounts.

In [ ]:
taus = np.linspace(-0.5, 0.5, 21)  # symbol periods
fig, ax = plt.subplots()
for rolloff in (0.1, 0.5):
    tx = ck.generate(
        C16,
        num_symbols=2**14,
        symbol_rate=32e9,
        sps=2,
        pulse=RRC(rolloff=rolloff),
        rng=1,
        device=DEVICE,
    )
    mf = ck.filtering.matched_filter(tx)
    classical, power = [], []
    for tau in taus:
        x = host(ck.timing.fft_fractional_delay(mf, delay=-2 * tau).samples)
        x = x / np.sqrt(np.mean(np.abs(x) ** 2))
        y, m = x[0::2], x[1::2]  # m[k] lies between y[k] and y[k + 1]
        classical.append(np.mean(np.real(m[:-1] * np.conj(y[:-1] - y[1:]))))
        p_y, p_m = np.abs(y) ** 2, np.abs(m) ** 2
        power.append(np.mean(-p_m[:-1] * (p_y[:-1] - p_y[1:])))
    ax.plot(taus, classical, label=f"classical, rolloff {rolloff}")
    ax.plot(taus, power, "--", label=f"power, rolloff {rolloff}")
ax.set_xlabel("Timing error [symbol periods]")
ax.set_ylabel("Mean detector output")
ax.legend()
plt.show()

Both detectors are zero at the right timing and negative when late.  At a
rolloff of 0.5 the classical S-curve is the steeper; at 0.1 it has shrunk
to a fifth of that while the power form has grown.

## 3. Tracking the clock

`Gardner()` puts the detector in a second-order loop.  The loop sets the
strobe spacing to `sps · (1 + ν + μ e)`; the integrator `ν` settles at the
clock offset.  Before it starts, the S-curve is measured open loop on the
first 2048 symbols: its amplitude scales the gains to the requested
`normalized_bandwidth` (`B·T`, default `1e-3`) and its zero starts the loop
at the right phase.

`estimate_sampling_clock_offset` returns the offset in ppm, from the slope
of the **timing trajectory** `τ_k` (the strobe of symbol `k` in symbol
periods, minus `k`), and the trajectory itself.  The trajectory climbs by
6.6 symbol periods over the record; plotted is its error against the timing
the link imposed, `2k(1 + ε)/2 − k = kε`.

In [ ]:
fig, ax = plt.subplots()
for method in ("classical", "power"):
    est = ck.timing.estimate_sampling_clock_offset(drifting, Gardner(method=method))
    traj = host(est.trajectory)
    k = np.arange(traj.size)
    true = 2 * k * (1 + 50e-6) / 2 - k
    print(
        f"{method:9s}: {float(est.value):7.3f} ppm (true 50), "
        f"detector gain {float(est.detector_gain):.2f}"
    )
    ax.plot(traj - true, label=method)
ax.set_xlabel("Symbol")
ax.set_ylabel("Timing error [symbol periods]")
ax.legend()
plt.show()

What remains is the loop's **jitter**, about a hundredth of a symbol: the
detector's own noise, mostly pattern noise from the data rather than AWGN,
filtered by the loop.  A narrower bandwidth averages more of it but follows
a changing clock more slowly.

`correct_sampling_clock_offset` with a `Gardner` resamples the record at
`sps` instants per tracked symbol, with the band-limited interpolator of
`apply_sampling_clock_offset`: the drift and the static timing are gone, and
the symbol-rate slicer sees clean symbols over the whole record.  What the
jitter leaves is ISI, `σ_τ² · Σ_{n≠0} g'(n)²` for the raised-cosine `g`.

In [ ]:
for method in ("classical", "power"):
    recovered = ck.timing.correct_sampling_clock_offset(
        drifting, Gardner(method=method)
    )
    sym = ck.multirate.decimate_to_symbol_rate(recovered)
    print(f"{method:9s}: SNR {ck.metrics.snr(sym, num_skip_symbols=1000):.2f} dB")
ck.plotting.plot_constellation(
    sym.samples[-20000:], title="After clock recovery, last 20000 symbols", show=True
)

## 4. A known offset

The clock offset of a setup changes slowly, so an estimate from one
capture serves the next.  `correct_sampling_clock_offset` with a
`SamplingClockOffsetEstimate` resamples by the constant factor `1 + ε`: the
drift goes, a static timing phase stays, which an adaptive equalizer
([17](17-adaptive-equalization.ipynb)) absorbs with the rest of the
channel.

In [ ]:
est = ck.timing.estimate_sampling_clock_offset(drifting, Gardner(method="power"))
next_capture = link(rolloff=0.1, ppm=50.0, delay=1.1)
fixed = ck.timing.correct_sampling_clock_offset(next_capture, est)
residual = ck.timing.estimate_sampling_clock_offset(fixed, Gardner(method="power"))
print(f"residual offset after the static correction: {float(residual.value):.3f} ppm")

**Summary**

* An ADC clock off by `ε` makes the symbols drift by `ε` symbol periods per
  symbol; a capture of 10⁵ symbols at tens of ppm drifts by whole symbols.
* Gardner's detector needs two samples per symbol and excess bandwidth; its
  power form keeps working at small rolloffs.
* `estimate_sampling_clock_offset` measures the offset, and
  `correct_sampling_clock_offset` resamples onto the symbol grid, tracking
  with `Gardner()` or removing a known offset.

**Next:** [13 · Frequency-offset estimation](13-frequency-offset-estimation.ipynb).